# PVpy tutorial notebook

The goal of this package is to offer an accessible and effecient tool in python (using only numpy and scipy) to perform both **symbolic** and **numeric** computations of 1-loop amplitude in QFT. 

The PVpy package contains two different layers that can be used independantly or together:

1. **Symbolic layer** 
<p>The first layer consist in the writting of loop integrals and perform their tensorial reduction in terms of Passarino-Veltman (PV) functions. Then the expression must be simplified and the precise kinematic conditions must be set. Then the symbolic expressions can be reduced into the simplest scalar PV functions and/or expanded into closed expression, allowing analytical computation with the use of sympy.

2. **Numeric layer** 
<p> The second layer consist in the use of the different PV functions as numpy callable. This part can be used independantly of the first one. The PV functions are defined vey carefully such that the numeric output is valid wether on the domain of definition. In all generality, the returned result is complex, eventhough the imaginary part is 0 when the threshold is respected (see the guide).

**Limitations**:
>So far only one-point, two-points and 3-points functions (A-functions, B-functions, C-functions) are implemented. The extension with box diagrams and four-points (D-functions) will be done later.

>Only the derivative w.r.t to $p^2$ of the two-points functions is implemented.

>The symbolic expression for three-points functions is not available for every cases of $C_0$ but the numerical procedure is valid for every kinematic condition.
 


# 1 Introduction to PVpy tensorial  notations

PVpy uses sympy for the symbolic part but needs to implement its own conventions for the Lorentz indices, Minkoswki metric, gamma matrices and Trace.

First the different functions need can be simply imported with

In [3]:
# Specific modules for the algebra need to be imported
from pvpy import slash, Gamma, Gamma5, DiracTrace, Dot, Mom, g, Eps, contract

 For symbolic computations one always need specific symbols for momentums, masses, Lorentz indices.. A set of predifinied symbols  is already part of PVpy and can be imported. The function `Dot(symb,symb)` can be used to create Lorentz scalar product of two symbols of a quadrivector.

In [4]:
from pvpy import k, p, p_1, p_2, p_3, m, m_1, m_2, m_3, mu, nu, rho, sigma, p2, p_12, p_22
import sympy as sp
display(k, p_1, m_1)

# Note that p2 or p_12 are shorthand notations for Dot(p,p) and Dot(p_1,p_1)
display( Dot(p,p))
display( p2)

k

p_1

m_1

p^2

p^2

If needed, new sympy symbols can be introduced, note that some symbols are already used by PVpy internaly, notably $d$ and $g$ which stand for the dimension of the space and the metric respectively


In [5]:
import sympy as sp
# New symbol xi
xi = sp.Symbol("xi")
xi

xi

Tensorial expression with metric or momentum can be easily build with the use of the functions `Mom(k, mu)` `Dot(k, ...)` `g(mu, nu)`, for instance
$$ g^{\mu\nu} + (1-\xi) \frac{k^{\mu}k^{\nu}}{k^2-\xi m^2} $$
can be written as

In [6]:
expr =  g(mu, nu) + (1-xi) * Mom(k,mu) * Mom(k, nu) / (Dot(k,k) - xi * m**2)
expr

(1 - xi)*k^{mu}*k^{nu}/(-m**2*xi + k^2) + g^{munu}

Similarly expression containing Dirac matrices, can be written with the use of `slash(symbol)`, `Gamma(indice)` functions, as an exemple 
$$ (p_1^{\sigma}\gamma_{\sigma} - m) \gamma^{\mu} (p_2^{\rho}\gamma_{\rho}-m) \gamma^{\nu} $$
can be defined as

In [7]:
expr   = (slash(p_1) - m) * Gamma(mu) * (slash(p_2) - m) * Gamma(nu)
expr

m**2*γ^{mu}*γ^{nu} - m*γ^{mu}*p_2̸*γ^{nu} - m*p_1̸*γ^{mu}*γ^{nu} + p_1̸*γ^{mu}*p_2̸*γ^{nu}

The trace of expressions containing Dirac matrices can be performed using the `DiracTrace(expression)` functions

In [8]:
DiracTrace(expr)

4*m**2*g^{munu} - 4*(p_1.p_2)*g^{munu} + 4*p_1^{mu}*p_2^{nu} + 4*p_1^{nu}*p_2^{mu}

Expressions with up to four free indices can be used

In [9]:
# Up to 4 free indices can be set
expr = Gamma(mu) * Gamma(nu) * Gamma(rho) * Gamma(sigma)
display(expr)
display( DiracTrace(expr) )

γ^{mu}*γ^{nu}*γ^{rho}*γ^{sigma}

4*g^{munu}*g^{rhosigma} - 4*g^{murho}*g^{nusigma} + 4*g^{musigma}*g^{nurho}

For chiral structure the $\gamma_5$ can be written with the `Gamma5()` function. For instance here we show the computation of the $Z \rightarrow ff$ width computation


$(p_1^{\sigma}\gamma_{\sigma} + m) \gamma^{\mu} (g_V - g_A \gamma_5) (p_2^{\rho}\gamma_{\rho}+m)(g_V + g_A \gamma_5) \gamma^{\nu}$ 

In [10]:
# Define new symbols for Vector and Axial couplings
g_V, g_A = sp.symbols('g_V g_A')

# Don't forget the empty parenthesis on Gamma5()
expr = (slash(p_1) + m) * Gamma(mu) * (g_V - g_A * Gamma5() )  * (slash(p_2) - m) * (g_V + g_A * Gamma5() ) * Gamma(nu) 
display(expr)
display(DiracTrace(expr))

g_A**2*m**2*γ^{mu}*γ₅**2*γ^{nu} - g_A**2*m*γ^{mu}*γ₅*p_2̸*γ₅*γ^{nu} + g_A**2*m*p_1̸*γ^{mu}*γ₅**2*γ^{nu} - g_A**2*p_1̸*γ^{mu}*γ₅*p_2̸*γ₅*γ^{nu} - g_A*g_V*m*γ^{mu}*γ₅*p_2̸*γ^{nu} + g_A*g_V*m*γ^{mu}*p_2̸*γ₅*γ^{nu} - g_A*g_V*p_1̸*γ^{mu}*γ₅*p_2̸*γ^{nu} + g_A*g_V*p_1̸*γ^{mu}*p_2̸*γ₅*γ^{nu} - g_V**2*m**2*γ^{mu}*γ^{nu} + g_V**2*m*γ^{mu}*p_2̸*γ^{nu} - g_V**2*m*p_1̸*γ^{mu}*γ^{nu} + g_V**2*p_1̸*γ^{mu}*p_2̸*γ^{nu}

4*g_A**2*m**2*g^{munu} - 4*g_A**2*(p_1.p_2)*g^{munu} + 4*g_A**2*p_1^{mu}*p_2^{nu} + 4*g_A**2*p_1^{nu}*p_2^{mu} + 8*I*g_A*g_V*eps^{mu,nu,p_1,p_2} - 4*g_V**2*m**2*g^{munu} - 4*g_V**2*(p_1.p_2)*g^{munu} + 4*g_V**2*p_1^{mu}*p_2^{nu} + 4*g_V**2*p_1^{nu}*p_2^{mu}

>**Caution** <p>Eventhough PVpy can show symbolic expressions in a Latex form with the use of the `display()` function, it should be used only as an intermediate way of checking the expression. Internally, to perform the Lorentz and Clifford algebras, the expression are expanded and thus the displaying is often unreadable.
Thus for a plaisant use, it is better to define small expression and then combine them together with the use of the `contract()` and the usual multiplication *

<p> For instance we can

$$  (p_1^{\sigma}\gamma_{\sigma} + m) \gamma^{\mu} (g_V - g_A \gamma_5) (p_2^{\rho}\gamma_{\rho}+m)(g_V + g_A \gamma_5) \gamma^{\nu} \left(-g^{\mu\nu} +  \frac{k^{\mu}k^{\nu}}{k^2}\right) $$


>**Caution** <p> The `DiracTrace()` function must be used on expression containing Dirac matrices **before** trying to contract it with an expression containg free Lorentz indices

In [11]:
expr_1 = (slash(p_1) + m) * Gamma(mu) * (g_V - g_A * Gamma5() )  * (slash(p_2) - m) * (g_V + g_A * Gamma5() ) * Gamma(nu) 
expr_2 =  -g(mu, nu) + Mom(k,mu) * Mom(k, nu) / Dot(k,k)

# The Dirac trace need to be done before trying to contract with another expression
expr = contract( DiracTrace(expr_1) * expr_2)
expr

-4*d*g_A**2*m**2 + 4*d*g_A**2*(p_1.p_2) + 4*d*g_V**2*m**2 + 4*d*g_V**2*(p_1.p_2) + 4*g_A**2*m**2 - 12*g_A**2*(p_1.p_2) + 8*g_A**2*(k.p_1)*(k.p_2)/k^2 - 4*g_V**2*m**2 - 12*g_V**2*(p_1.p_2) + 8*g_V**2*(k.p_1)*(k.p_2)/k^2

To substitute kinematic condition, one can use the `set_kinematic(expression, { Dot(k,k) : m_Z**2, ...})` function. From the example above, we wish to do following replacement:

$$\begin{aligned}
& p_1^2 = p_2^2 = m^2 \\
& k^2 = m_Z^2 \\
& p_1 \cdot p_2 = \frac{1}{2}(m_Z^2 - 2m^2)
\end{aligned}$$


For pure tree-level or already-finite expressions like the $Z \rightarrow f\bar{f} $ trace example here, it's always safe to substitute immediately.

In [12]:
from pvpy import set_kinematics

m_Z = sp.Symbol('m_Z', positive=True)
d = sp.Symbol('d')

# Z -> ff kinematics:

result = contract(DiracTrace(expr_1) * expr_2)
result = set_kinematics(result, {
    Dot(k, k)    : m_Z**2,
    Dot(p_1, p_2): (m_Z**2 - 2*m**2) / 2,
    Dot(k, p_1)  : m_Z**2 / 2,
    Dot(k, p_2)  : m_Z**2 / 2,
    d  : 4}) # or equivalently: result.subs(d, 4)

display(sp.simplify(result))

-16*g_A**2*m**2 + 4*g_A**2*m_Z**2 + 8*g_V**2*m**2 + 4*g_V**2*m_Z**2

The fully antisymetric Levi-civita symbol $\epsilon$ is defined as well, it can used with the Eps() function as

In [13]:
# Levi - civita symbol
expr = Eps(mu,nu, rho, sigma)
display(expr)
expr = contract( expr * Mom(p, mu) * Mom(p, nu) )
display(expr)

eps^{mu,nu,rho,sigma}

0

# 2 Computing 1-loop integrals

## 2.1  Propagators

A loop integrand is built from propagators of the form

$$D_i = (k + q_i)^2 - m_i^2$$

where $k$ is the loop momentum and $q_i$ are linear combinations of 
external momenta.  By convention, **the first propagator always has 
$q_0 = 0$**, this is always possible by choosing the routing of $k$.

Propagators are created with `Propagator(shift, mass)`:


In [14]:
from pvpy import Propagator, LoopIntegral, ReduceGeneralNumerator
from pvpy.symbols import k, m, p, p_1, p_2, m_0, m_1, m_2

# Each propagator is inserted in a list

#  one propagator  D_0 = k^2 - m_0^2
prop_A = [Propagator(0, m_0)]

# two propagators  D_0 = k^2 - m_0^2,  D_1 = (k-p_1)^2 - m_1^2
prop_B = [Propagator(0, m_0),
          Propagator(-p_1, m_1)]

# three propagators
prop_C = [Propagator(0, m_0),
          Propagator(-p_1, m_1),
          Propagator(p_1 + p_2, m_2)]


## 2.2  Defining a loop integral

A loop integral is assembled from a list of propagators and a numerator.
The numerator is a sympy expression written in terms of `Dot(k,...)`, 
`Mom(k, mu)`, and any external tensors `g`, `Mom`, couplings...).
For a **scalar** integral the numerator is simply `1`.

`ReduceGeneralNumerator(loop, numerator)` performs the full Passarino-Veltman reduction and 
returns a sympy expression in terms of PV scalar functions.

In [15]:
# --- Scalar 1-point function ---
loop_A = LoopIntegral(k, prop_A)
A0_result = ReduceGeneralNumerator(loop_A, sp.Integer(1))
display(A0_result)


# --- Scalar 2-points functions ---
loop_B = LoopIntegral(k, prop_B)
B0_result = ReduceGeneralNumerator(loop_B, sp.Integer(1))
display(B0_result)


A0(m_0)

B0(p_1**2, m_0, m_1)

In the general case, we whish to have more complicated case, where the numerator is a tensorial expression. For instance, we can look at the one loop vaccum polarization in QED:
$$ i\Pi_{\mu\nu} = -\mu^{\epsilon}\int \frac{\mathrm{d}^dk}{(2\pi)^d}\mathrm{Tr}\left[ie\gamma_{\mu}\frac{i(\gamma_{\rho}k^{\rho}+m)}{k^2-m^2}ie\gamma_{\nu}\frac{i(\gamma_{\sigma}(k-p)^{\sigma}+m)}{(k-p)^2-m^2}\right].$$

In [16]:
# --- QED vacuum polarization: fermion self-energy ---

# Propagators: D0 = k² - m0²,  D1 = (k - p)² - m1²
prop = [Propagator(0, m_0), Propagator(-p, m_1)]
loop = LoopIntegral(k, prop)

# Numerator: Tr[γ^μ (k̸ + m0) γ^ν (k̸ - p̸ + m1)]
numerator = DiracTrace(Gamma(mu) * (slash(k) + m_0) * Gamma(nu) * (slash(k - p) + m_1))

# Tensorial decomposition: T^{μν} = A g^{μν} + B p^μ p^ν
result = ReduceGeneralNumerator(loop, numerator)
display(result)

-6*m_0**2*g^{munu}*B0(p**2, m_0, m_1) + 4*m_0*m_1*g^{munu}*B0(p**2, m_0, m_1) + 2*m_1**2*g^{munu}*B0(p**2, m_0, m_1) + 2*p**2*g^{munu}*B0(p**2, m_0, m_1) - 8*p^{mu}*p^{nu}*B1(p**2, m_0, m_1) + 8*p^{mu}*p^{nu}*B11(p**2, m_0, m_1) - 2*g^{munu}*A0(m_0) - 2*g^{munu}*A0(m_1) + 8*g^{munu}*B00(p**2, m_0, m_1)

We see that the result is fully expressed in term of PV functions and tensorial objects. However, most of the time only a scalar coefficient is needed, in our case from a rank-2 tensor result which in generality writes $$T^{\mu\nu} = A\,g^{\mu\nu} + B\,p^\mu p^\nu$$ one might need only to extract the longitudinal or transverse part.

To do so, the `Project(expr, mu, nu, p, mode)` function can be used as follows:

| `mode` | returns | meaning |
|--------|---------|---------|
| `'g'` or `'T'` | $A$ | coefficient of $g^{\mu\nu}$ (= transverse scalar) |
| `'pp'` | $B$ | coefficient of $p^\mu p^\nu$ |
| `'L'` | $A + p^2 B$ | longitudinal scalar |


In [17]:
from pvpy import Project, PV_reduce

Pi_T = Project(result, mu, nu, p, 'T')   # transverse 
Pi_L = Project(result, mu, nu, p, 'L')   # longitudinal

print("Transvere part of the 1-loop fermionic self-energy in QED:")
display(Pi_T)

Transvere part of the 1-loop fermionic self-energy in QED:


-6*m_0**2*B0(p**2, m_0, m_1) + 4*m_0*m_1*B0(p**2, m_0, m_1) + 2*m_1**2*B0(p**2, m_0, m_1) + 2*p**2*B0(p**2, m_0, m_1) - 2*A0(m_0) - 2*A0(m_1) + 8*B00(p**2, m_0, m_1)

## 2.3 Simplifying and setting kinematics

**`PV_simplify`** applies safe forward identities that avoid $1/p^2$ denominators and replace the $d = 4 -\epsilon$ safely, taking into account the pole part of each PV functions that could be invovled.
Always run it before `set_kinematics`.

**`set_kinematics`** substitutes any kinematic condition, in particular, it is very important, that if one or many of the variable are $0$ to specify it directly in `set_kinematics`.

> **Important:** 

> substitute `p**2` (algebraic square), **not** `pvpy.symbols.p2`
> (a separate named Symbol `p^2` — different object).
> To set a variable as $0$, one can just use the normal numeric $0$ without creating a new symbol. Assisde from $0$, **never** use  float or numeric to replace a symbol! 

In [18]:
from pvpy import PV_simplify, set_kinematics

# PV_simplify (here is the expression doesn't change because no expression could be simplified)
Pi_T_s = PV_simplify(Pi_T)
print("Simplified expression:\n(it is normal that the expression didn't change if nothing could be simplified)")
display(Pi_T_s)

# define a symbolic mass for the Z boson
m_Z = sp.Symbol('m_Z', positive = True) 

# Set kinematics: equal fermion mass and p² = m_Z²
Pi_T_kin = set_kinematics(Pi_T_s, {p**2: m_Z**2, m_0: m, m_1: m})
Pi_T_kin_1 = set_kinematics(Pi_T_s, {p**2: 0, m_0: m, m_1: m})
Pi_T_kin_2 = set_kinematics(Pi_T_s, {p**2: m_Z**2, m_0: 0, m_1: 0})
# -> care to use p**2 or (p_1**2 if the symbol is p_1) and NOT p^2

# Here the expression is simplified because of the kinematic
print("Expression after the setting of the kinematic, p**2 is non-zero:")
display(Pi_T_kin)

print("Expression after the setting of the kinematic, p**2 is zero:")
display(Pi_T_kin_1)
 
print("Expression after the setting of the kinematic, m_1 = m_0 = 0:")
display(Pi_T_kin_2)

Simplified expression:
(it is normal that the expression didn't change if nothing could be simplified)


-6*m_0**2*B0(p**2, m_0, m_1) + 4*m_0*m_1*B0(p**2, m_0, m_1) + 2*m_1**2*B0(p**2, m_0, m_1) + 2*p**2*B0(p**2, m_0, m_1) - 2*A0(m_0) - 2*A0(m_1) + 8*B00(p**2, m_0, m_1)

Expression after the setting of the kinematic, p**2 is non-zero:


2*m_Z**2*B0(m_Z**2, m, m) - 4*A0(m) + 8*B00(m_Z**2, m, m)

Expression after the setting of the kinematic, p**2 is zero:


-4*A0(m) + 8*B00(0, m, m)

Expression after the setting of the kinematic, m_1 = m_0 = 0:


2*m_Z**2*B0(m_Z**2, 0, 0) + 8*B00(m_Z**2, 0, 0)

## Taking the derivative of two-points functions wrt to $p^2$

The derivative of the two-points (B-functions), is often needed in physical computation. That is why the derivativ with respect to $p^2$ of each B-function is defined. Thus if needed, using the normal sympy command `sp.diff(expr, p2)`, `p2` here is the symbol of the squared momentum $p^2$

## 2.4 Two paths: numeric or symbolic

After `PV_simplify` and `set_kinematics` the expression is ready for evaluation and two paths are available for the user.



### 2.4.1  **Symbolic path**

 Using the **`.doit(part = "full")`** method, the user can express each PV function in their full expression (according to the kinematic set before). This can result in very lenghty result because the PV functions are tiedous functions with a lot of logarithm. The **`part`** can be set as follows:


| `part` |  meaning |
|--------|---------|
| `'pole'` |  returns **only** the pole part in the form $a\cdot \frac{1}{\bar{\epsilon}}$|
| `'finite'` |  returns **only** the finite part of the expression (care the $\mu$ term are part of it) |
| `'full'` | returns the full expression i.e pole + finite |


Because the full `.doit(part)` are often hard to manipulate, it can be better before to try to reduce the expression in term of simpler PV functions. The is the goal of **`PV_reduce(expr)`** which express each PV function in terms of scalar ones. After the user can still use the `.doit(part)` method if needed.

The use of `set_kinematic` and `PV_reduce` together enables to check simplification which could not occur when formulas are not reduced. For instance the **Ward identity** which for a gauge-invariant amplitude equal-mass fermion loop in QED is $$ \Pi_L = A + p^2 B = 0$$ can be checked after setting $m_0 = m_1 = m$ and reducing the expression.

> **Important:**

> The user can also define directly symbolic expression with PV functions of its choice. For this, one juste need to import the functions as `from pvpy import A0, B0, B1, ...`

> The derivative of the one-point and two-points functions w.r.t $p^2$ are implemented. To obtain such function, one just need to call the usual `sp.diff(expr, p2)` from sympy, it will act smoothly on PV_functions object. Care second order derivative are not implemented. If the user want to differentiate the expression, they need to call `.doit()` before


In [19]:
# --- Symbolic path: the doit( part = ' ') method--- #

# RECALL: set_kinematic must ALWAYS be called before and enable to set p² = 0 safely 


# .doit() method, expand directly the chosen part in a closed-form expressions 
# part can 'full', 'pole', 'finite'
print('Closed expression of the pole transverse part:\n')
display(Pi_T_kin.doit(part = 'pole'))

# Usual function from sympy can then used
print('Simplified closed expression of the pole transverse part:\n')
display(sp.simplify(Pi_T_kin.doit(part = 'pole')))

print('Closed expression of the finite transverse part:\n')
display(Pi_T_kin.doit(part = 'finite'))

print('Closed expression of the full transverse part:\n')
display(Pi_T_kin.doit(part = 'full'))

Closed expression of the pole transverse part:



-4*m**2/\bar{\varepsilon} + 2*m_Z**2/\bar{\varepsilon} + 8*(m**2/2 - m_Z**2/12)/\bar{\varepsilon}

Simplified closed expression of the pole transverse part:



4*m_Z**2/(3*\bar{\varepsilon})

Closed expression of the finite transverse part:



-8*m**2*(1 - log(m**2/mu**2))/3 + 8*m**2*(sqrt(-4*m**2/m_Z**2 + 1)*log((2*m**2 + m_Z**2*sqrt(-4*m**2/m_Z**2 + 1) - m_Z**2)/(2*m**2)) - log(m**2/mu**2) + 2)/3 + 8*m**2/3 - 4*m_Z**2*(sqrt(-4*m**2/m_Z**2 + 1)*log((2*m**2 + m_Z**2*sqrt(-4*m**2/m_Z**2 + 1) - m_Z**2)/(2*m**2))/2 - log(m**2/mu**2)/2 + 1)/3 + 2*m_Z**2*(sqrt(-4*m**2/m_Z**2 + 1)*log((2*m**2 + m_Z**2*sqrt(-4*m**2/m_Z**2 + 1) - m_Z**2)/(2*m**2)) - log(m**2/mu**2) + 2) - 4*m_Z**2/9

Closed expression of the full transverse part:



-8*m**2*(-log(m**2/mu**2) + 1 + 1/\bar{\varepsilon})/3 + 8*m**2*(sqrt(-4*m**2/m_Z**2 + 1)*log((2*m**2 + m_Z**2*sqrt(-4*m**2/m_Z**2 + 1) - m_Z**2)/(2*m**2)) - log(m**2/mu**2) + 2 + 1/\bar{\varepsilon})/3 + 8*m**2/3 - 4*m_Z**2*(sqrt(-4*m**2/m_Z**2 + 1)*log((2*m**2 + m_Z**2*sqrt(-4*m**2/m_Z**2 + 1) - m_Z**2)/(2*m**2))/2 - log(m**2/mu**2)/2 + 1 + 1/(2*\bar{\varepsilon}))/3 + 2*m_Z**2*(sqrt(-4*m**2/m_Z**2 + 1)*log((2*m**2 + m_Z**2*sqrt(-4*m**2/m_Z**2 + 1) - m_Z**2)/(2*m**2)) - log(m**2/mu**2) + 2 + 1/\bar{\varepsilon}) - 4*m_Z**2/9

In [20]:
# --- Symbolic path: PV_reduce function --- #

from pvpy import PV_reduce

# Use PV_reduce to get an expression in terms of simpler PV functions
Pi_T_red = PV_reduce(Pi_T_kin)
print('Reduced expression of the transverse part:\n')
display(Pi_T_red)

# .doit() can still be applied and give naturally the same result
print('Closed expression of the transverse part, applying doit on the reduced one:\n')
display(Pi_T_red.doit(part = 'full'))

print('Closed expression of the transverse part, can be simplified using sympy:\n')
display(sp.simplify(Pi_T_red.doit(part = 'full')))

# Example: Ward identity check, Pi_L must vanish at equal masses 
# -> need PV_reduce first so the algebraic cancellations between B functions happen

Pi_L_check = set_kinematics(PV_reduce(Pi_L), {m_0: m, m_1: m})
print("Ward identity Pi_L at m0=m1=m (should be 0):")
display(Pi_L_check)

Reduced expression of the transverse part:



8*m**2*B0(m_Z**2, m, m)/3 + 8*m**2/3 + 4*m_Z**2*B0(m_Z**2, m, m)/3 - 4*m_Z**2/9 - 8*A0(m)/3

Closed expression of the transverse part, applying doit on the reduced one:



-8*m**2*(-log(m**2/mu**2) + 1 + 1/\bar{\varepsilon})/3 + 8*m**2*(sqrt(-4*m**2/m_Z**2 + 1)*log((2*m**2 + m_Z**2*sqrt(-4*m**2/m_Z**2 + 1) - m_Z**2)/(2*m**2)) - log(m**2/mu**2) + 2 + 1/\bar{\varepsilon})/3 + 8*m**2/3 + 4*m_Z**2*(sqrt(-4*m**2/m_Z**2 + 1)*log((2*m**2 + m_Z**2*sqrt(-4*m**2/m_Z**2 + 1) - m_Z**2)/(2*m**2)) - log(m**2/mu**2) + 2 + 1/\bar{\varepsilon})/3 - 4*m_Z**2/9

Closed expression of the transverse part, can be simplified using sympy:



16*m**2/3 + 8*m**2*sqrt(-4*m**2 + m_Z**2)*log(1 - m_Z**2/(2*m**2) + m_Z*sqrt(-4*m**2 + m_Z**2)/(2*m**2))/(3*m_Z) - 4*m_Z**2*log(m**2/mu**2)/3 + 20*m_Z**2/9 + 4*m_Z*sqrt(-4*m**2 + m_Z**2)*log(1 - m_Z**2/(2*m**2) + m_Z*sqrt(-4*m**2 + m_Z**2)/(2*m**2))/3 + 4*m_Z**2/(3*\bar{\varepsilon})

Ward identity Pi_L at m0=m1=m (should be 0):


0

**Why use PV_reduce before doit()?**

>The key reason is when algebraic cancellations happen. Without `PV_reduce`, calling `doit()` expands each PV function independently into its full closed form i.e a combination of logarithms and dilogarithms and then asks sympy to cancel logarithms against each other, which sp.expand alone rarely achieves. With `PV_reduce` first, the derived functions (B1, B11, B00, …) are replaced by algebraic combinations of the primitives A0 and B0 before any logarithm appears. The cancellations then happen at the level of opaque symbolic objects, which sympy handles effortlessly. Only once the expression is fully reduced in terms of A0 and B0 does doit() expand those simple primitives, producing a much more compact result. In short, `PV_reduce` provides the structured, physics-informed simplification that a generic `sp.simplify()` call would attempt blindly at the cost of being slow and unreliable on logarithm-heavy expressions. The pattern `PV_reduce -> doit()` is therefore both faster and more predictable than `doit()` alone.

In [21]:
# --- Create directly an expression using PV functions --- #

# Import the needed function, derivative can also be imported direclty as
from pvpy import A0, B0, B1, B11, B00, dB0_dp2, dB00_dp2

# Define the expression
expr = A0(m) + B0(p2, m_0,m_1) + B0(0,m,m) + B00(p2,0,m)
print("Expression involving PV function, defined directly as such:")
display(expr)


# Take the derivative w.r.t p²
expr_diff = sp.diff(expr,p2)
print("Expression involving derivative of the B-function w.r.t p²:")
display(expr_diff)

# Equivalently, one can directly define derivative as 
expr_diff_2 = dB0_dp2(p2, m_0,m_1) + dB00_dp2(p2,0,m)
print("Second and same expression involving derivative of the B-function w.r.t p²:")
display(expr_diff_2)

Expression involving PV function, defined directly as such:


A0(m) + B0(0, m, m) + B0(p^2, m_0, m_1) + B00(p^2, 0, m)

Expression involving derivative of the B-function w.r.t p²:


dB00_dp2(p^2, 0, m) + dB0_dp2(p^2, m_0, m_1)

Second and same expression involving derivative of the B-function w.r.t p²:


dB00_dp2(p^2, 0, m) + dB0_dp2(p^2, m_0, m_1)

### 2.4.2 **Numeric path**
 The second path is the numeric path, with the use of `**compile(expr,part,mu)**`, a PV object can transformed in a numpy callable ready for to be called with arrays. Each PV function is safely defined and computed numerically such that wether the combination of variable that is given, the correct result is returned even in degenerated case like $m_0 = m_1$ or $p^2 = 0$. Still it must be used after `set_kinematic(expr)` and `PV_simplify(expr)` such that simplification can be done in the numeric kernel. When calling compile the list of the argument to give to the numpy callable will be printed as an ouput.

**Scale dependancy**: most PV function contain a scale dependant part which involves the scale $\mu$; In QFT, observable don't depend on the scale value, however for intermediate calculation it must be taken into account. That is why when calling `compile(expr,part,**mu**)` , $\mu$ must be fixed at some value. By default it is fixed to one, but we **encourage** the user to try different values and to check that their physical result are scale dependant.

| `part` |  meaning |
|--------|---------|
| `'pole'` |  returns **only** the coefficient $a$ in front of the pole part which the form $a\cdot \frac{1}{\bar{\epsilon}}$|
| `'finite'` |  returns **only** the finite part of the expression (scale dependant terms containing $\mu$ are part of it) |


> **Never** call `compile` after `PV_reduce`: the reduced expression might contains
> $1/p^2$ denominators and $1/\bar\varepsilon$ poles that the numeric integrator
> cannot evaluate.

> If one the variable is constant, it needs to be implemended specificaly exterior the package, for instance `mZ = m_Z_val * np.ones(N)

> For specific values of the kinematic, it absolutely normal to get a complex result, this expected. (See the appendix of the guide for more detail)

In [22]:
# --- Numeric path --- #

# compile: turns the PV-function expression into a numpy-callable.
# The callable's argument order is shown automatically.

from pvpy.numeric import compile
import numpy as np

# Set the mu scale, and choose which part to compute
scale = 91 # EW scale
f = compile(Pi_T_kin, part = "finite", mu = scale)  

# Evaluate at one point: the signature is (m, m_Z) alphabetically
# m_Z = 91.2 GeV,  m (electron) = 0.000511 GeV
m_val   = 0.000511
m_Z_val = 91.2
print("Pi_T(m_Z², m, m) =", f(m_val,m_Z_val))

# Arrays of the same shap can be given as input
m_val = np.array([0, 1, 2]) # the masses are always positive or null
m_Z_val = np.array([30, 60, 90]) # p² the momentum can positive or negativ(timelike or spacelike)
print("Result with input arrays:\n", f(m_val,m_Z_val))

# Expressions defined previous directly with PV functions can also be used:
f1 = compile(expr, part = 'finite', mu = scale )
f1_diff = compile(expr_diff, part = 'finite', mu = scale )

**Callable signature:**

(m, m_Z)

Pi_T(m_Z², m, m) = [18434.50739908+34840.0112009j]
Result with input arrays:
 [ 4663.18918713 +3769.91118431j 12006.56312943+15079.63775333j
 18270.80553146+33929.15098112j]


**Callable signature:**

(p^2, m, m_0, m_1)

**Callable signature:**

(p^2, m, m_0, m_1)

**The workflow can be summarized as follow**


```
PV_simplify → set_kinematics ─┬─ compile      (numpy callable, fast numerics)
                               └─ PV_reduce    (try symbolic reduction )
                                      └─ .doit()  (expand to closed forms)
```


# 3. Physical example for symbolic and numeric usage

## 3.1 Symbolic example: Closed expression of the oblique parameters S,T,U for a $SU(2)_L \times U(1)_Y$ left doublet and right singlet

## 3.2 Numerical example: computation of the S,T,U parameters in the SM, and check with the analytical expression from 3.1